# Here i will make an ETL that extract elecricity costs data from a website and make a file out of it

# and later make an example on how to make the ETL run after a certain and automate it

In [29]:
# importing needed libraries
# pandas to deal with tables, requests to make HTTP API calls

import pandas as pd
import requests

In [30]:
# Extract

# define the extract function
# extract the 2 columns about for country and electricity price

from bs4 import BeautifulSoup

url = "https://worldpopulationreview.com/country-rankings/cost-of-electricity-by-country"

response = requests.get(url)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

# find table rows
rows = soup.select("table tbody tr")

data = []
for row in rows:
    cols = row.find_all("td")
    if len (cols) >= 2:
       country = cols[1].get_text(strip=True)
       cost = cols[2].get_text(strip=False)
       data.append((country,cost))

df = pd.DataFrame(data, columns=["Country", "ELECTRCITY COST PER KWH 2024 (USD)"])
df

,Country,ELECTRCITY COST PER KWH 2024 (USD)
0,Bermuda,$0.46
1,Italy,$0.46
2,Ireland,$0.44
3,Cayman Islands,$0.43
4,Liechtenstein,$0.41
...,...,...
141,Sudan,$0.01
142,Syria,$0.01
143,Cuba,$0.01
144,Ethiopia,$0


In [31]:
# transform
# define the transform function

# add id column
df.insert(0, 'id', range(1, len(df) + 1))
# rename columns and rop null and duplicates, and make electricity column float
df = df.rename(columns={"COUNTRY": "Country", "ELECTRCITY COST PER KWH 2024 (USD)": "electrcity per kwh 2024 (USD)"})
df = df.dropna()
df = df.drop_duplicates()
df["electrcity per kwh 2024 (USD)"] = df["electrcity per kwh 2024 (USD)"].str.replace("$", "", regex = False).astype(float)
print(df.shape)
print(df.head)

(146, 3)
<bound method NDFrame.head of       id         Country  electrcity per kwh 2024 (USD)
0      1         Bermuda                           0.46
1      2           Italy                           0.46
2      3         Ireland                           0.44
3      4  Cayman Islands                           0.43
4      5   Liechtenstein                           0.41
..   ...             ...                            ...
141  142           Sudan                           0.01
142  143           Syria                           0.01
143  144            Cuba                           0.01
144  145        Ethiopia                           0.00
145  146            Iran                           0.00

[146 rows x 3 columns]>


In [32]:
# Load
# loading the extracted and cleaned data into an excle and in AWS

excel_path = "Final_gas_prices.xlsx"
df.to_excel(excel_path, index=False)
print("the excel file", excel_path)

the excel file Final_gas_prices.xlsx


In [33]:
df = pd.read_excel("Final_gas_prices.xlsx")
df.head(5)

,id,Country,electrcity per kwh 2024 (USD)
0,1,Bermuda,0.46
1,2,Italy,0.46
2,3,Ireland,0.44
3,4,Cayman Islands,0.43
4,5,Liechtenstein,0.41


# set the code for a specific time and automate
# here i will make the code again but make it an automated process

In [34]:
# install schudule library used to automate a process
!pip install schedule

In [35]:
# importing needed libraries

# time to set time of the execution of a process
# pandas to deal with tables, requests to make HTTP API calls
# BeautifulSoup to help in parsing data

import time
import pandas as pd
import requests
import schedule
from bs4 import BeautifulSoup

In [36]:
# Extract
# define the extract function
# extract the 2 columns about for country and electricity price

def extract():
    url = "https://worldpopulationreview.com/country-rankings/cost-of-electricity-by-country"
    response = requests.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    rows = soup.select("table tbody tr")

    data = []
    for row in rows:
       cols = row.find_all("td")
       if len (cols) >= 2:
          country = cols[1].get_text(strip=True)
          cost = cols[2].get_text(strip=False)
          data.append((country,cost))

    df = pd.DataFrame(data, columns=["COUNTRY", "ELECTRCITY COST PER KWH 2024 (USD)"])
    return(df)

In [37]:
# transform
# define the transform function

def transform(df):
    # add id column
    df.insert(0, 'id', range(1, len(df) + 1))
    # rename columns and rop null and duplicates, and the cost a column a float
    df = df.rename(columns={"COUNTRY": "Country", "ELECTRCITY COST PER KWH 2024 (USD)": "electrcity per kwh 2024 (USD)"})
    df = df.dropna()
    df = df.drop_duplicates()
    df["electrcity per kwh 2024 (USD)"] = df["electrcity per kwh 2024 (USD)"].str.replace("$", "", regex = False).astype(float)
    return df

In [38]:
# Load
# loading the extracted and cleaned data into an excel

def load(df):
    excel_path = "Final_gas_prices2.xlsx"
    df.to_excel(excel_path, index=False)
    print("the excel file", excel_path)

In [39]:
# run the ETL pipeline
def run_pipeline():
    df = extract()
    df = transform(df)
    load(df)
    print("ETL cycle complete.\n")

In [40]:
# executing the ETL pipeline after 10 seconds
run_pipeline()
time.sleep(10)

the excel file Final_gas_prices2.xlsx
ETL cycle complete.



In [41]:
# make the ETL pipeline run each 1 minute
schedule.every(1).minutes.do(run_pipeline)

while True:
    schedule.run_pending()

the excel file Final_gas_prices2.xlsx
ETL cycle complete.

the excel file Final_gas_prices2.xlsx
ETL cycle complete.

the excel file Final_gas_prices2.xlsx
ETL cycle complete.

the excel file Final_gas_prices2.xlsx
ETL cycle complete.

the excel file Final_gas_prices2.xlsx
ETL cycle complete.

the excel file Final_gas_prices2.xlsx
ETL cycle complete.

the excel file Final_gas_prices2.xlsx
ETL cycle complete.

the excel file Final_gas_prices2.xlsx
ETL cycle complete.

the excel file Final_gas_prices2.xlsx
ETL cycle complete.

the excel file Final_gas_prices2.xlsx
ETL cycle complete.

the excel file Final_gas_prices2.xlsx
ETL cycle complete.

the excel file Final_gas_prices2.xlsx
ETL cycle complete.



KeyboardInterrupt: 